In [2]:
import pandas as pd
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 300)
pd.options.mode.chained_assignment = None  # default='warn'

In [4]:
df1 = pd.read_parquet('/volume/ECG_tokenizer/output/mimic_test_qa_10k.parquet')

In [6]:
mimic_path = '/media/data1/datasets/MIMIC-IV/Diagnosis/mimic_labelbox_bert_v4_all.parquet'
mimic_df = pd.read_parquet(mimic_path)
    

In [7]:
display(mimic_df.head(n=5))

,gender,age_at_ecg,diagnosis,npy_path,rr_interval,p_onset,p_end,qrs_onset,qrs_end,t_end,p_axis,qrs_axis,t_axis,new_PatientID,Bifid,RaVL + SV3 > 28 mm (H) or 20 mm (F),"Hyperacute T wave (lateral, V5-V6)","Hyperacute T wave (septal, V1-V2)",ST depression et T inversion in V5 or V6,Large >0.08 s,"Hyperacute T wave (anterior, V3-V4)",Auricular bigeminy,Biphasic,Ventricular bigeminy,J wave,...,Early repolarization,Ventricular Rhythm,Irregularly irregular,Atrial tachycardia (>= 100 BPM),R complex in V5-V6,"ST elevation (lateral - I, aVL, V5-V6)",Brugada,Bi-atrial enlargement,"Q wave (lateral- I, aVL, V5-V6)",ST upslopping,"T wave inversion (inferior - II, III, aVF)",Regularly irregular,Bradycardia,"qRS in V5-V6-I, aVL",Q wave (anterior - V3-V4),Acute MI,ST depression (anterior - V3-V4),Right ventricular hypertrophy,T wave inversion (septal- V1-V2),ST downslopping,Left bundle branch block,Low voltage,U wave,Left atrial enlargement,pid
0,F,52,Sinus rhythm;Possible right atrial abnormality;Borderline ECG,/media/data1/ravram/MIMIC-IV/1.0/files/p1000/p10000032/s40689238/40689238.npy,659,40,128,170,258,518,81,77,79,10000032,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11000000
1,F,52,Sinus rhythm;Possible right atrial abnormality;Borderline ECG,/media/data1/ravram/MIMIC-IV/1.0/files/p1000/p10000032/s44458630/44458630.npy,722,40,124,162,246,504,77,75,70,10000032,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11000000
2,F,52,Sinus tachycardia;Normal ECG except for rate,/media/data1/ravram/MIMIC-IV/1.0/files/p1000/p10000032/s49036311/49036311.npy,600,40,130,162,244,474,79,72,77,10000032,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11000000
3,F,55,Sinus rhythm;Normal ECG,/media/data1/ravram/MIMIC-IV/1.0/files/p1000/p10000117/s45090959/45090959.npy,659,40,146,180,254,538,79,66,69,10000117,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11000001
4,F,57,Sinus rhythm,/media/data1/ravram/MIMIC-IV/1.0/files/p1000/p10000117/s48446569/48446569.npy,659,368,29999,504,590,868,84,80,77,10000117,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11000001


In [9]:

# Prepare mimic_df for merging
mimic_df['npy_id'] = mimic_df['npy_path'].str.extract(r'/([^/]+)\.npy$')[0]
demographic_cols = ['new_PatientID', 'npy_id', 'gender', 'age_at_ecg', 'rr_interval']
available_cols = [col for col in demographic_cols if col in mimic_df.columns]

# Define input paths for test and train sets (from generate_train_test_datasets.py)
test_input = '/media/data1/datasets/ECG_Tokenizer/parquets/test/mimic_mhi_psa_test_updated_with_questions.parquet'
train_input = '/media/data1/datasets/ECG_Tokenizer/parquets/train/mimic_mhi_psa_train_updated_with_questions.parquet'

# Load test set and merge
try:
    df_test = pd.read_parquet(test_input)
    if 'waveform_name' in df_test.columns:
        df_test['npy_id'] = df_test['waveform_name'].str.replace('.npy', '')
    else:
        df_test['npy_id'] = df_test.index.astype(str)
    df_test_merged = df_test.merge(
        mimic_df[available_cols],
        on='npy_id',
        how='left',
        suffixes=('', '_mimic')
    )
    print("Test set merged shape:", df_test_merged.shape)
    display(df_test_merged.head())
except Exception as e:
    print(f"Could not load or merge test set: {e}")

# Load train set and merge
try:
    df_train = pd.read_parquet(train_input)
    if 'waveform_name' in df_train.columns:
        df_train['npy_id'] = df_train['waveform_name'].str.replace('.npy', '')
    else:
        df_train['npy_id'] = df_train.index.astype(str)
    df_train_merged = df_train.merge(
        mimic_df[available_cols],
        on='npy_id',
        how='left',
        suffixes=('', '_mimic')
    )
    print("Train set merged shape:", df_train_merged.shape)
    display(df_train_merged.head())
except Exception as e:
    print(f"Could not load or merge train set: {e}")


Test set merged shape: (100031, 88)


,waveform_path_psa,report,Sinusal,Regular,Monomorph,QS complex in V1-V2-V3,R complex in V5-V6,"T wave inversion (inferior - II, III, aVF)",Left bundle branch block,RaVL > 11 mm,SV1 + RV5 or RV6 > 35 mm,"T wave inversion (lateral -I, aVL, V5-V6)",T wave inversion (anterior - V3-V4),Left axis deviation,Left ventricular hypertrophy,Bradycardia,"Q wave (inferior - II, III, aVF)",Afib,Irregularly irregular,Atrial tachycardia (>= 100 BPM),Nonspecific intraventricular conduction delay,Premature ventricular complex,Polymorph,T wave inversion (septal- V1-V2),Right bundle branch block,...,Q wave (anterior - V3-V4),ST upslopping,Right superior axis,Ventricular tachycardia,ST elevation (posterior - V7-V8-V9),Ectopic atrial rhythm (< 100 BPM),Lead misplacement,Third Degree AV Block,Acute MI,Early repolarization,Q wave (posterior - V7-V9),Bi-atrial enlargement,LV pacing,Brugada,Ventricular Rhythm,no_qrs,dataset,waveform_name,waveform_path_original,question,npy_id,new_PatientID,gender,age_at_ecg,rr_interval
0,/media/data1/datasets/MIMIC-IV/adjusted_signals/test/40000079.npy,Sinus rhythm;Inferior and anterior ST-T changes are nonspecific;Borderline ECG,1,1,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,mimic,40000079.npy,/media/data1/ravram/MIMIC-IV/1.0/files/p1636/p16368287/s40000079/40000079.npy,What does this ECG tell you about my heart?,40000079,16368287,M,68,909
1,/media/data1/datasets/MIMIC-IV/adjusted_signals/test/40000144.npy,Sinus tachycardia;Normal ECG except for rate,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,mimic,40000144.npy,/media/data1/ravram/MIMIC-IV/1.0/files/p1414/p14144725/s40000144/40000144.npy,What do my ECG readings indicate?,40000144,14144725,F,53,555
2,/media/data1/datasets/MIMIC-IV/adjusted_signals/test/40000152.npy,Sinus rhythm;Anterior T wave changes are nonspecific;Borderline ECG,1,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,mimic,40000152.npy,/media/data1/ravram/MIMIC-IV/1.0/files/p1608/p16089780/s40000152/40000152.npy,What does this ECG reveal about my heart’s function?,40000152,16089780,F,56,645
3,/media/data1/datasets/MIMIC-IV/adjusted_signals/test/40000240.npy,Sinus rhythm;Probable left ventricular hypertrophy,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,mimic,40000240.npy,/media/data1/ravram/MIMIC-IV/1.0/files/p1833/p18332475/s40000240/40000240.npy,Would you mind explaining what my ECG readings entail?,40000240,18332475,M,34,741
4,/media/data1/datasets/MIMIC-IV/adjusted_signals/test/40000305.npy,Sinus tachycardia;Left axis deviation;IV conduction defect;Possible anterior infarct - age undetermined;Possible lateral infarct - age undetermined;Abnormal ECG,1,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,mimic,40000305.npy,/media/data1/ravram/MIMIC-IV/1.0/files/p1024/p10246670/s40000305/40000305.npy,How should I interpret this ECG?,40000305,10246670,M,44,550


Train set merged shape: (350997, 88)


,waveform_path_psa,report,Sinusal,Regular,Monomorph,QS complex in V1-V2-V3,R complex in V5-V6,"T wave inversion (inferior - II, III, aVF)",Left bundle branch block,RaVL > 11 mm,SV1 + RV5 or RV6 > 35 mm,"T wave inversion (lateral -I, aVL, V5-V6)",T wave inversion (anterior - V3-V4),Left axis deviation,Left ventricular hypertrophy,Bradycardia,"Q wave (inferior - II, III, aVF)",Afib,Irregularly irregular,Atrial tachycardia (>= 100 BPM),Nonspecific intraventricular conduction delay,Premature ventricular complex,Polymorph,T wave inversion (septal- V1-V2),Right bundle branch block,...,Q wave (anterior - V3-V4),ST upslopping,Right superior axis,Ventricular tachycardia,ST elevation (posterior - V7-V8-V9),Ectopic atrial rhythm (< 100 BPM),Lead misplacement,Third Degree AV Block,Acute MI,Early repolarization,Q wave (posterior - V7-V9),Bi-atrial enlargement,LV pacing,Brugada,Ventricular Rhythm,no_qrs,dataset,waveform_name,waveform_path_original,question,npy_id,new_PatientID,gender,age_at_ecg,rr_interval
0,/media/data1/datasets/MIMIC-IV/adjusted_signals/train/40000035.npy,Sinus rhythm;Possible inferior infarct - age undetermined;Abnormal ECG,1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,mimic,40000035.npy,/media/data1/ravram/MIMIC-IV/1.0/files/p1659/p16598616/s40000035/40000035.npy,How do you view this ECG result?,40000035,16598616,F,56,800
1,/media/data1/datasets/MIMIC-IV/adjusted_signals/train/40000115.npy,A-V dual-paced rhythm with some inhibition,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,mimic,40000115.npy,/media/data1/ravram/MIMIC-IV/1.0/files/p1257/p12576058/s40000115/40000115.npy,How do you see my ECG results?,40000115,12576058,F,93,779
2,/media/data1/datasets/MIMIC-IV/adjusted_signals/train/40000162.npy,Demand pacing;Pacemaker rhythm - no further analysis;Abnormal ECG,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,mimic,40000162.npy,/media/data1/ravram/MIMIC-IV/1.0/files/p1376/p13767422/s40000162/40000162.npy,Can you explain the ECG results and whether they indicate any problems with my heart's rhythm or function?,40000162,13767422,M,80,952
3,/media/data1/datasets/MIMIC-IV/adjusted_signals/train/40000172.npy,Sinus rhythm;Normal ECG,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,mimic,40000172.npy,/media/data1/ravram/MIMIC-IV/1.0/files/p1443/p14431564/s40000172/40000172.npy,What should I understand from this ECG?,40000172,14431564,M,38,722
4,/media/data1/datasets/MIMIC-IV/adjusted_signals/train/40000175.npy,Sinus rhythm;Normal ECG,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,mimic,40000175.npy,/media/data1/ravram/MIMIC-IV/1.0/files/p1360/p13605623/s40000175/40000175.npy,Explain what this ECG reading shows.,40000175,13605623,M,67,869


In [10]:
# Find how many unique new_PatientID in train are also in test
train_pids = set(df_train_merged['new_PatientID'].dropna().unique())
test_pids = set(df_test_merged['new_PatientID'].dropna().unique())
common_pids = train_pids & test_pids
print(f"Number of unique new_PatientID in train: {len(train_pids)}")
print(f"Number of unique new_PatientID in test: {len(test_pids)}")
print(f"Number of new_PatientID in train that are also in test: {len(common_pids)}")


Number of unique new_PatientID in train: 94640
Number of unique new_PatientID in test: 26773
Number of new_PatientID in train that are also in test: 0
